# Module 1.3 — Multi-Head Attention, MLP & the Block
**DeskMate SLM Curriculum · Phase 1 · Notebook 04**

Run on **Google Colab** or **Kaggle** (free T4 or CPU).

By the end you will have:
- `MultiHeadAttention`, `FeedForward`, and `Block` classes with verified shapes
- A stacked multi-block model with val loss beating the Module 1.2 baseline
- Noticeably more coherent generated text than the bigram model

Read `1.3_multihead_block.md` first for the theory.

---


## Step 0 — Install & Seed

In [ ]:
%%capture
!pip install -q torch==2.3.1 matplotlib==3.9.0


In [ ]:
import random, math, pathlib, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")


## Step 1 — Runtime, Paths & Corpus

In [ ]:
try:
    import google.colab; RUNTIME = "colab"
    PROJECT_ROOT = pathlib.Path("/content/slm")
except ImportError:
    try:
        import kaggle_secrets; RUNTIME = "kaggle"
        PROJECT_ROOT = pathlib.Path("/kaggle/working/slm")
    except ImportError:
        RUNTIME = "local"
        PROJECT_ROOT = pathlib.Path(".").resolve()

DATA_DIR   = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Runtime : {RUNTIME}")


In [ ]:
corpus_path = DATA_DIR / "tiny_shakespeare.txt"
if not corpus_path.exists():
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    urllib.request.urlretrieve(url, corpus_path)

text = corpus_path.read_text(encoding="utf-8")
chars  = sorted(set(text))
V      = len(chars)
stoi   = {c: i for i, c in enumerate(chars)}
itos   = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: "".join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

B, T = 32, 64

def get_batch(split):
    src = train_data if split == "train" else val_data
    ix  = torch.randint(len(src) - T, (B,))
    x   = torch.stack([src[i   : i+T  ] for i in ix])
    y   = torch.stack([src[i+1 : i+T+1] for i in ix])
    return x.to(device), y.to(device)

print(f"Vocab {V}  Train {len(train_data):,}  Val {len(val_data):,}")


## Step 2 — `Head` (Carry-over from Module 1.2)

We redefine `Head` here so the notebook is self-contained.
One change: `forward` returns only the output (not the weights) for clean composition.


In [ ]:
class Head(nn.Module):
    # Single causal self-attention head.
    def __init__(self, d_model, d_head, block_size, dropout=0.1):
        super().__init__()
        self.d_head = d_head
        self.q = nn.Linear(d_model, d_head, bias=False)
        self.k = nn.Linear(d_model, d_head, bias=False)
        self.v = nn.Linear(d_model, d_head, bias=False)
        self.drop = nn.Dropout(dropout)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)
        scores  = q @ k.transpose(-2, -1) * self.d_head**-0.5
        scores  = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        weights = self.drop(scores.softmax(dim=-1))
        return weights @ v


## Step 3 — `MultiHeadAttention`

Run `n_heads` `Head` modules in parallel, concatenate along the last dimension,
then project back to `d_model`. The projection mixes information across heads.


In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        d_head = d_model // n_heads
        self.heads = nn.ModuleList([
            Head(d_model, d_head, block_size, dropout) for _ in range(n_heads)
        ])
        self.proj = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        # Each head → (B, T, d_head); cat along last dim → (B, T, d_model)
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.drop(self.proj(out))


In [ ]:
# Shape verification
d_model, n_heads = 64, 4
mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads, block_size=T).to(device)
x_test = torch.randn(B, T, d_model, device=device)
out_mha = mha(x_test)
print(f"Input  : {x_test.shape}")
print(f"Output : {out_mha.shape}   (must be same as input)")
assert out_mha.shape == x_test.shape, "shape mismatch!"
print("Shape assertion passed.")

n_params_mha = sum(p.numel() for p in mha.parameters())
print(f"MHA parameters: {n_params_mha:,}")


In [ ]:
# Show that n_heads parallel narrow heads == 1 wide head in param count
d_head = d_model // n_heads
params_per_head = 3 * d_model * d_head   # Q, K, V each (d_model × d_head)
params_all_heads = params_per_head * n_heads
params_proj = d_model * d_model
print(f"d_model={d_model}, n_heads={n_heads}, d_head={d_head}")
print(f"Q+K+V per head: {params_per_head:,}  × {n_heads} heads = {params_all_heads:,}")
print(f"Output proj   : {params_proj:,}")
print(f"Total QKV+proj: {params_all_heads + params_proj:,}")
print()
print("One wide head (d_model x d_model for each of Q,K,V + proj):")
print(f"  {3 * d_model * d_model + d_model * d_model:,}  — identical param count, different inductive bias")


## Step 4 — `FeedForward` MLP

Position-wise: the same MLP applied independently to every token.
`d_model → 4*d_model → GELU → d_model`.
Attention mixes *which* tokens contribute; the MLP transforms *within* each position.


In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, 4 * d_model),
            nn.GELU(),
            nn.Linear(4 * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
ff = FeedForward(d_model).to(device)
out_ff = ff(x_test)
print(f"FeedForward  input : {x_test.shape}")
print(f"FeedForward  output: {out_ff.shape}   (same shape)")
assert out_ff.shape == x_test.shape
print("Shape assertion passed.")

n_params_ff = sum(p.numel() for p in ff.parameters())
print(f"FeedForward parameters: {n_params_ff:,}  (4x expansion: {d_model} → {4*d_model} → {d_model})")


## Step 5 — `Block`: Pre-Norm + Residuals

Combine `MultiHeadAttention` and `FeedForward` with pre-LayerNorm and residual connections.

```
x = x + Attention(LayerNorm(x))   ← attention sub-layer
x = x + MLP(LayerNorm(x))         ← MLP sub-layer
```

The residual stream `x` flows straight through; each sub-layer adds a correction.


In [ ]:
class Block(nn.Module):
    # One transformer block: pre-norm MHA + FF, both with residuals.
    def __init__(self, d_model, n_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, block_size, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


In [ ]:
block = Block(d_model=d_model, n_heads=n_heads, block_size=T).to(device)
out_block = block(x_test)
print(f"Block input : {x_test.shape}")
print(f"Block output: {out_block.shape}   (same shape — residual stream preserved)")
assert out_block.shape == x_test.shape
print("Shape assertion passed.")

n_params_block = sum(p.numel() for p in block.parameters())
print(f"Block parameters: {n_params_block:,}")
print(f"  = MHA {n_params_mha:,} + FF {n_params_ff:,} + 2×LayerNorm {2*2*d_model:,}")


In [ ]:
# Stack 4 blocks and measure total
n_layers = 4
blocks_stack = nn.Sequential(*[Block(d_model, n_heads, T) for _ in range(n_layers)]).to(device)
out_stack = blocks_stack(x_test)
print(f"Stack of {n_layers} blocks → output: {out_stack.shape}")
n_params_stack = sum(p.numel() for p in blocks_stack.parameters())
print(f"Stack parameters: {n_params_stack:,}  ({n_params_stack/1e3:.1f}k)")


## Step 6 — Full Model & Training

Wire N `Block`s between token/position embeddings and a final LayerNorm + LM head.
Train for 3000 steps; compare val loss to the Module 1.2 single-head baseline.


In [ ]:
N_LAYERS  = 4
D_MODEL   = 64
N_HEADS   = 4
DROPOUT   = 0.1
MAX_STEPS = 3_000
LR        = 1e-3

class NanoGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, block_size, dropout):
        super().__init__()
        self.tok_emb  = nn.Embedding(vocab_size, d_model)
        self.pos_emb  = nn.Embedding(block_size, d_model)
        self.drop_emb = nn.Dropout(dropout)
        self.blocks   = nn.Sequential(*[
            Block(d_model, n_heads, block_size, dropout) for _ in range(n_layers)
        ])
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        tok = self.tok_emb(idx)
        pos = self.pos_emb(torch.arange(T, device=idx.device))
        x   = self.drop_emb(tok + pos)
        x   = self.blocks(x)
        x   = self.ln_final(x)
        return self.lm_head(x)                          # (B, T, vocab_size)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -T:]
            logits   = self(idx_cond)[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = logits.softmax(dim=-1)
            nxt   = torch.multinomial(probs, num_samples=1)
            idx   = torch.cat([idx, nxt], dim=1)
        return idx


model = NanoGPT(V, D_MODEL, N_HEADS, N_LAYERS, T, DROPOUT).to(device)
n_total = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_total:,}  ({n_total/1e3:.1f}k)")


In [ ]:
@torch.no_grad()
def estimate_loss(eval_iters=100):
    model.eval()
    results = {}
    for split in ("train", "val"):
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            xb, yb = get_batch(split)
            logits = model(xb)
            losses[k] = F.cross_entropy(logits.view(-1, V), yb.view(-1))
        results[split] = losses.mean().item()
    model.train()
    return results

opt = torch.optim.AdamW(model.parameters(), lr=LR)
train_losses, val_losses, steps_logged = [], [], []
EVAL_INTERVAL = 500

torch.manual_seed(SEED)
init = estimate_loss()
print(f"Initial  →  train: {init['train']:.4f}  val: {init['val']:.4f}")


In [ ]:
for step in range(MAX_STEPS):
    xb, yb = get_batch("train")
    logits  = model(xb)
    loss    = F.cross_entropy(logits.view(-1, V), yb.view(-1))
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        est = estimate_loss()
        train_losses.append(est["train"])
        val_losses.append(est["val"])
        steps_logged.append(step)
        print(f"step {step:5d}  train: {est['train']:.4f}  val: {est['val']:.4f}")

print("Training complete.")


In [ ]:
ATTENTION_LM_VAL = 2.40   # approx val loss from Module 1.2 — update with your value

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(steps_logged, train_losses, label="train (4-block NanoGPT)", color="#4C72B0")
ax.plot(steps_logged, val_losses,   label="val   (4-block NanoGPT)", color="#DD8452", linestyle="--")
ax.axhline(ATTENTION_LM_VAL, color="grey", linestyle=":", label=f"Module 1.2 val baseline ≈ {ATTENTION_LM_VAL}")
ax.set_xlabel("Step"); ax.set_ylabel("Loss"); ax.set_title("4-Block NanoGPT vs Single-Head Baseline")
ax.legend(); plt.tight_layout()
plt.savefig(str(MODELS_DIR / "block_loss_curve.png"), bbox_inches="tight")
plt.show()
print(f"Final val loss : {val_losses[-1]:.4f}  (baseline: {ATTENTION_LM_VAL})")


## Step 7 — Generate Text

In [ ]:
torch.manual_seed(SEED)
context = torch.zeros((1, 1), dtype=torch.long, device=device)

print("=== temperature=1.0 (default sampling) ===")
out1 = decode(model.generate(context, 400, temperature=1.0)[0].tolist())
print(out1)


In [ ]:
print("=== temperature=0.7, top_k=20 (more focused) ===")
out2 = decode(model.generate(context, 400, temperature=0.7, top_k=20)[0].tolist())
print(out2)
print()
print("Note: with only 65 chars vocab and 4 small blocks, text looks Shakespearean-ish")
print("but not coherent. Module 1.4 trains longer with more capacity.")


In [ ]:
# Save checkpoint
ckpt_path = MODELS_DIR / "block_ckpt.pt"
torch.save({
    "model_state": model.state_dict(),
    "config": {
        "d_model": D_MODEL, "n_heads": N_HEADS,
        "n_layers": N_LAYERS, "block_size": T, "vocab_size": V,
    },
}, ckpt_path)
print(f"Saved: {ckpt_path}  ({ckpt_path.stat().st_size / 1024:.1f} KB)")


## Checkpoint

> **What do residual connections buy you as depth grows?**

Write your answer below. A strong answer covers:
1. The vanishing/exploding gradient problem without residuals (Jacobian multiplication).
2. The identity shortcut: `∂loss/∂x_k = ∂loss/∂x_{k+1} × (I + ∂f/∂x_k)`.
3. The practical consequence: layers that aren't useful learn near-zero corrections and become identity pass-throughs, so adding depth never hurts.


In [ ]:
answer = """
[Write your answer here]
"""
print(answer)


## Deliverable Summary

| Artifact | Location |
|---|---|
| Loss curve (4-block vs baseline) | `models/block_loss_curve.png` |
| Trained checkpoint | `models/block_ckpt.pt` |

**What you've built:** the stackable `Block` — the repeating unit of every transformer.
`MultiHeadAttention` + `FeedForward` + pre-norm + residuals.

**Next:** Module 1.4 — Assemble nano-SLM, train properly, and generate.
You'll train the same architecture for longer with larger capacity and implement
temperature + top-k sampling cleanly. The result is your first complete language model.
